# Daily Practice — 2026-11-24 — ML Testing & MLOps: Data Contracts and Reproducibility Testing for a Diagnostic Classifier

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset)
— 569 real patient records, loaded via `sklearn.datasets.load_breast_cancer`. Each row is 30
real-valued measurements (radius, texture, perimeter, area, smoothness, compactness, concavity,
concave points, symmetry, fractal dimension — each as a mean, standard-error, and "worst" value)
digitized from fine-needle-aspirate biopsy images, with a binary malignant/benign target. This is
the real UCI diagnostic dataset, not a synthetic or toy one.

## Problem statement

In traditional SQA, you don't ship code without contract tests on its inputs and outputs, and you
don't trust a build that isn't reproducible. ML pipelines need the exact same discipline, just aimed
at data and at randomness instead of at function arguments: a silently malformed input batch can
retrain a model into something worse without any code change at all, and a "successful" training run
that can't be reproduced is a build you can't trust, debug, or roll back with confidence.

You are the QA gate for a hospital's diagnostic-assist pipeline. Before any new batch of patient scan
measurements is allowed to retrain or score the production model, it has to pass through your checks.
Your job is to build that gate.

## What you should produce

1. `build_bounds(df_ref, cols)` — derives per-feature acceptable ranges from a trusted reference
   (the training set), not hardcoded magic numbers.
2. `validate_schema`, `validate_ranges`, `validate_missingness` — three data-contract checks, each
   returning a list of human-readable violation strings (empty list = pass).
3. A deliberately corrupted copy of a held-out batch, used to confirm each validator actually fires
   on the specific problem it's meant to catch (a schema-only test that never sees bad data isn't
   a test).
4. `run_reproducibility_test(...)` — trains the same model + seed multiple times and asserts the
   predictions are **bit-for-bit identical** every run.
5. `seed_sensitivity_test(...)` — trains across several *different* seeds and reports how much the
   F1 score moves, so you can tell real seed-to-seed instability apart from a model that's actually
   broken.
6. `ship_gate(...)` — combines all of the above into one pass/fail decision with reasons attached.
7. A short write-up: did any validator fire on genuinely clean data? If so, what does that mean for
   how you'd tune it before putting it in front of an on-call engineer?

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

RANDOM_STATE = 42

## Load the data

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns=lambda c: c.replace(" ", "_"))
FEATURE_COLS = [c for c in df.columns if c != "target"]

X = df[FEATURE_COLS]
y = df["target"]

# X_train/X_test play two roles below: X_train is the trusted "reference" batch you derive
# data contracts from, and X_test stands in for a new incoming batch you need to validate.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
df.head()

## Your task

Fill in the TODOs below. Function signatures and the overall shape are given; the logic is not.

In [ ]:
def build_bounds(df_ref, cols, lower_q=0.0, upper_q=1.0, margin=0.05):
    '''TODO: for each column, compute the (lower_q, upper_q) quantiles of df_ref[col],
    then widen the [lo, hi] interval by `margin` * (hi - lo) on each side.
    Return a dict: {column_name: (lo, hi)}.
    '''
    raise NotImplementedError

In [ ]:
def validate_schema(frame, expected_columns, expected_dtype_kind="f"):
    '''TODO: return a list of violation strings covering:
      - any column in expected_columns missing from frame
      - any column in frame not in expected_columns
      - any expected column whose dtype.kind != expected_dtype_kind
    Return [] if everything is fine.
    '''
    raise NotImplementedError


def validate_ranges(frame, bounds):
    '''TODO: for each (col, (lo, hi)) in bounds, find rows of frame[col] outside [lo, hi]
    and append one violation string per offending column (not per row) stating the count
    and the bound, e.g. "column mean_radius: 2 rows out of [5.92, 29.17]".
    '''
    raise NotImplementedError


def validate_missingness(frame, max_null_fraction=0.0):
    '''TODO: for each column, compute the fraction of nulls; append a violation string
    for any column whose null fraction exceeds max_null_fraction.
    '''
    raise NotImplementedError

## Prove the validators actually catch something

TODO: build `bounds` from `X_train` with `build_bounds`. Run all three validators against the
clean `X_test` batch and print the results. Then make a corrupted copy of `X_test` — inject at
least one out-of-range value, one wildly-out-of-range value, and one null — and run the
validators again to confirm each problem is caught.

In [ ]:
# TODO: bounds = build_bounds(X_train, FEATURE_COLS)
# TODO: print validate_schema / validate_ranges / validate_missingness on the clean X_test

# TODO: bad_batch = X_test.copy(); inject a negative value, an extreme value, and a NaN
# TODO: re-run the three validators on bad_batch and print the results


## Reproducibility and seed sensitivity

TODO: implement `run_reproducibility_test`, which trains `model_fn(seed)` on
`(X_train, y_train)` `n_runs` times with the *same* seed and checks that every run's
predictions on `X_test` are identical via `np.array_equal`. Return `(all_identical: bool,
list_of_prediction_arrays)`.

TODO: implement `seed_sensitivity_test`, which trains `model_fn(seed)` once per seed in
`seeds`, computes the test F1 score each time, and returns the list of scores.

In [ ]:
def run_reproducibility_test(model_fn, X_train, y_train, X_test, seed, n_runs=3):
    raise NotImplementedError


def seed_sensitivity_test(model_fn, X_train, y_train, X_test, y_test, seeds):
    raise NotImplementedError


# TODO: define model_fn = lambda seed: RandomForestClassifier(n_estimators=50, random_state=seed)
# TODO: call run_reproducibility_test with seed=7, n_runs=3 and print the result
# TODO: call seed_sensitivity_test with seeds=[1, 2, 3, 4, 5]; print the scores and their std


## The ship gate

TODO: implement `ship_gate`, which runs every check above against a given batch and model
setup and returns a single dict report: `{"passed": bool, "reasons": [...]}`. A batch fails
the gate if any data-contract validator returns violations, if the model isn't bit-reproducible
at a fixed seed, or if the seed-sensitivity F1 standard deviation exceeds `max_f1_std`.

In [ ]:
def ship_gate(frame, bounds, expected_columns, model_fn, X_train, y_train, X_test, y_test,
              seed=7, seeds=(1, 2, 3, 4, 5), max_f1_std=0.02):
    raise NotImplementedError

# TODO: run ship_gate on the clean X_test batch and on bad_batch; print both reports


### Write-up

TODO, in 3-5 sentences: did `validate_ranges` flag anything on the **clean** `X_test` batch before you corrupted it? If so, what caused it, and what would you change about `build_bounds` (or about how you'd deploy this gate) to keep that from paging someone at 3am for nothing?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Build reference bounds from the trusted training batch
def build_bounds(df_ref, cols, lower_q=0.0, upper_q=1.0, margin=0.05):
    bounds = {}
    for c in cols:
        lo = df_ref[c].quantile(lower_q)
        hi = df_ref[c].quantile(upper_q)
        span = hi - lo
        bounds[c] = (lo - margin * span, hi + margin * span)
    return bounds

# 2. Data-contract validators
def validate_schema(frame, expected_columns, expected_dtype_kind="f"):
    violations = []
    missing = set(expected_columns) - set(frame.columns)
    if missing:
        violations.append(f"missing columns: {sorted(missing)}")
    extra = set(frame.columns) - set(expected_columns)
    if extra:
        violations.append(f"unexpected columns: {sorted(extra)}")
    for c in expected_columns:
        if c in frame.columns and frame[c].dtype.kind != expected_dtype_kind:
            violations.append(f"column {c} has dtype {frame[c].dtype}, expected kind {expected_dtype_kind}")
    return violations

def validate_ranges(frame, bounds):
    violations = []
    for c, (lo, hi) in bounds.items():
        if c not in frame.columns:
            continue
        bad = frame[(frame[c] < lo) | (frame[c] > hi)]
        if len(bad) > 0:
            violations.append(f"column {c}: {len(bad)} rows out of [{lo:.4f}, {hi:.4f}]")
    return violations

def validate_missingness(frame, max_null_fraction=0.0):
    violations = []
    frac = frame.isnull().mean()
    bad = frac[frac > max_null_fraction]
    for c, f in bad.items():
        violations.append(f"column {c}: {f:.2%} null, exceeds {max_null_fraction:.2%}")
    return violations

bounds = build_bounds(X_train, FEATURE_COLS)
print("schema (clean):", validate_schema(X_test, FEATURE_COLS))
print("ranges (clean):", validate_ranges(X_test, bounds))
print("missing (clean):", validate_missingness(X_test))

# 3. Corrupt a copy and confirm each problem gets caught
bad_batch = X_test.copy()
bad_batch.loc[bad_batch.index[0], "mean_radius"] = -5.0
bad_batch.loc[bad_batch.index[1], "mean_area"] = 999999.0
bad_batch.loc[bad_batch.index[2], "mean_texture"] = np.nan

print("ranges (corrupted):", validate_ranges(bad_batch, bounds))
print("missing (corrupted):", validate_missingness(bad_batch))

# 4. Reproducibility + seed sensitivity
def run_reproducibility_test(model_fn, X_train, y_train, X_test, seed, n_runs=3):
    preds_list = []
    for _ in range(n_runs):
        model = model_fn(seed)
        model.fit(X_train, y_train)
        preds_list.append(model.predict(X_test))
    identical = all(np.array_equal(preds_list[0], p) for p in preds_list[1:])
    return identical, preds_list

def seed_sensitivity_test(model_fn, X_train, y_train, X_test, y_test, seeds):
    scores = []
    for s in seeds:
        model = model_fn(s)
        model.fit(X_train, y_train)
        p = model.predict(X_test)
        scores.append(f1_score(y_test, p))
    return scores

model_fn = lambda seed: RandomForestClassifier(n_estimators=50, random_state=seed)
identical, _ = run_reproducibility_test(model_fn, X_train, y_train, X_test, seed=7, n_runs=3)
print("deterministic with fixed seed:", identical)

scores = seed_sensitivity_test(model_fn, X_train, y_train, X_test, y_test, seeds=[1, 2, 3, 4, 5])
print("f1 scores across seeds:", scores, "std:", np.std(scores))

# 5. Ship gate
def ship_gate(frame, bounds, expected_columns, model_fn, X_train, y_train, X_test, y_test,
              seed=7, seeds=(1, 2, 3, 4, 5), max_f1_std=0.02):
    reasons = []
    reasons += [f"schema: {v}" for v in validate_schema(frame, expected_columns)]
    reasons += [f"range: {v}" for v in validate_ranges(frame, bounds)]
    reasons += [f"missing: {v}" for v in validate_missingness(frame)]

    identical, _ = run_reproducibility_test(model_fn, X_train, y_train, X_test, seed)
    if not identical:
        reasons.append("reproducibility: predictions differ across repeated runs at a fixed seed")

    f1_scores = seed_sensitivity_test(model_fn, X_train, y_train, X_test, y_test, seeds)
    f1_std = float(np.std(f1_scores))
    if f1_std > max_f1_std:
        reasons.append(f"seed sensitivity: F1 std {f1_std:.4f} exceeds max {max_f1_std}")

    return {"passed": len(reasons) == 0, "reasons": reasons}

print("clean batch gate:", ship_gate(X_test, bounds, FEATURE_COLS, model_fn, X_train, y_train, X_test, y_test))
print("bad batch gate:", ship_gate(bad_batch, bounds, FEATURE_COLS, model_fn, X_train, y_train, X_test, y_test))
```

**Typical output**

```
schema (clean): []
ranges (clean): ['column mean_smoothness: 2 rows out of [0.0584, 0.1488]', 'column worst_radius: 1 rows out of [6.6700, 34.3900]', 'column worst_perimeter: 1 rows out of [41.4655, 238.2445]', 'column worst_area: 1 rows out of [22.8600, 3594.3400]']
missing (clean): []
ranges (corrupted): ['column mean_radius: 1 rows out of [5.9245, 29.1664]', 'column mean_area: 1 rows out of [25.7250, 2616.7750]', 'column mean_smoothness: 2 rows out of [0.0584, 0.1488]', 'column worst_radius: 1 rows out of [6.6700, 34.3900]', 'column worst_perimeter: 1 rows out of [41.4655, 238.2445]', 'column worst_area: 1 rows out of [22.8600, 3594.3400]']
missing (corrupted): ['column mean_texture: 0.88% null, exceeds 0.00%']
deterministic with fixed seed: True
f1 scores across seeds: [0.9655172413793104, 0.9655172413793104, 0.9726027397260274, 0.9583333333333334, 0.9655172413793104] std: 0.004512446891445952
clean batch gate: {'passed': False, 'reasons': ["range: column mean_smoothness: 2 rows out of [0.0584, 0.1488]", "range: column worst_radius: 1 rows out of [6.6700, 34.3900]", "range: column worst_perimeter: 1 rows out of [41.4655, 238.2445]", "range: column worst_area: 1 rows out of [22.8600, 3594.3400]"]}
bad batch gate: {'passed': False, 'reasons': ["range: column mean_radius: 1 rows out of [5.9245, 29.1664]", "range: column mean_area: 1 rows out of [25.7250, 2616.7750]", "range: column mean_smoothness: 2 rows out of [0.0584, 0.1488]", "range: column worst_radius: 1 rows out of [6.6700, 34.3900]", "range: column worst_perimeter: 1 rows out of [41.4655, 238.2445]", "range: column worst_area: 1 rows out of [22.8600, 3594.3400]", "missing: column mean_texture: 0.88% null, exceeds 0.00%"]}
```

**Explanation**

`run_reproducibility_test` passes cleanly: `RandomForestClassifier` with a fixed `random_state` is
bit-reproducible across runs on the same machine, exactly like re-running a deterministic unit test —
if this ever came back `False` for a model you expected to be deterministic, that's a real bug (an
unseeded sub-step, multithreading nondeterminism, a library version drift) worth its own ticket.
`seed_sensitivity_test` shows F1 moving by well under a point across five different seeds
(std ≈ 0.0045) — normal, healthy seed noise, not a sign the model is broken.

The more interesting result is that `validate_ranges` **fails on the clean, untouched `X_test`
batch** — four columns have a handful of rows just outside the bounds derived from `X_train`. This
isn't a bug in the validator; it's a direct consequence of deriving contracts from a finite sample:
`X_train`'s min/max (even widened by a 5% margin) doesn't perfectly contain every value a same-
distribution test set can produce, especially for skewed features like `worst_area`. In a real
on-call rotation, a gate this strict would train someone to ignore its alerts within a week. The fix
isn't to loosen the validator until it never fires — that just hides the next *real* corruption — it's
to separate "boundary values a legitimately clean batch can occasionally produce" (handle with a
percentile-based margin tuned against held-out clean data, or a minimum-violation-count threshold
before paging) from "this is structurally impossible" (a negative radius, a value 100x anything ever
observed, a null in a feature that's never been null) — the second category is what `bad_batch`
demonstrates, and it's unambiguous in every run. That same false-positive-rate tuning problem is
exactly what makes flaky tests painful in traditional CI, just relocated to the data layer.

</details>